# S10 · Create the target structure

Creates the schemas and the empty Delta tables in the INTERNAL target catalog, one schema per run. **Structure only — no rows move.** Every table arrives empty.

Runs on compute rather than through the catalog CRUD API because that API returns `202 Accepted` and can create nothing, so a table 'created' that way cannot be verified.

---

*Generated from `engine/dataplane/01_create_structure.py` by `engine/target/stage_notebooks.py`. Regenerate with `snowmig.py build-notebooks`; do not hand-edit — an edit here is overwritten on the next build. Change the source instead.*

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────
# Edit these, then run the notebook top to bottom. Every value is a
# plain Python literal: `None` means the flag is not passed at all,
# and `True` means a bare switch is passed.
#
# Scope and mode are INPUTS. To migrate less, change a value here --
# never edit the stage logic below to make it cover less.
PARAMS = {
    'source-mode': 'connector',
    'source-config': None,
    'source-catalog': None,
    'target-catalog': None,  # REQUIRED
    'schema': None,
    'target-schema': None,
    'mode': 'ddl-plan',
    'ddl-plan': None,
    'reports-dir': None,
    'dry-run': False,
    'force': False,
}


def _argv(params, repeated=()):
    """PARAMS -> argv. None is omitted; True is a bare switch; a list
    follows its flag, or repeats the flag per value for a name in
    `repeated` (an append-style flag)."""
    argv = []
    for key, value in params.items():
        if value is None or value is False:
            continue
        if value is True:
            argv.append(f'--{key}')
            continue
        values = value if isinstance(value, (list, tuple)) else [value]
        if key in repeated:
            for v in values:
                argv.extend((f'--{key}', str(v)))
        else:
            argv.append(f'--{key}')
            argv.extend(str(v) for v in values)
    return argv


ARGV = _argv(PARAMS, repeated=('schema',))
print('arguments:', ARGV)

_missing = [k for k in ['target-catalog'] if not PARAMS.get(k)]
if _missing:
    raise ValueError(
        f'set these PARAMS before running: {_missing}')

## Shared source helpers

Inlined from `engine/dataplane/snowmig_source.py` so this notebook runs with nothing else uploaded beside it.

In [ ]:
"""How the migration scripts READ Snowflake from inside AIDP. Two modes.

LIVE-VERIFIED 2026-09-16 on a real cluster (Spark 3.5, AIDP 4.x):

  connector (default)  spark.read.format("aidataplatform") with
                       type=SNOWFLAKE. Talked to the account, read a table
                       (87 rows x 9 cols) and ran a pushdown query
                       (`current_user()` = the service user). Needs NO extra
                       cluster library — the format is built in — and needs
                       NO successful catalog crawl.

  external-catalog     three-part names `catalog.schema.table` against a
                       registered EXTERNAL catalog. Cheaper (no per-read
                       Snowflake login) but it can only see what the CRAWLER
                       has already discovered, and on the validated
                       deployment the crawler failed with
                       "CONNECTOR_0067 ... Login has timed out" while the
                       connector above worked with the same credentials.

So: connector mode is the default because it is the one proven end to end,
and external-catalog mode is kept for a deployment whose crawl succeeds.

The option NAMES are the live-verified raw ones, which differ from the
Python helper's keyword names — `user.name` (not `user`), `database.name`,
`authentication.method` ∈ {Basic, KeyPair}, `private.key.content`. A wrong
name fails loud (`DATA_ACCESS_LAYER_0001 - Required spark option ... was not
provided`), which is how these were established.

Credentials come from a CONFIG FILE, never from arguments: the same rule the
control plane follows. Point --source-config at a JSON file shaped like
snowmig-config.example.yaml (JSON, on the workspace mount).
"""
from __future__ import annotations

import json
import pathlib

__all__ = ["SOURCE_MODES", "SourceConfigError", "SnowflakeSource",
           "load_source_config"]

SOURCE_MODES = ("connector", "external-catalog")

AIDP_FORMAT = "aidataplatform"


# The verbs this transport may send. Deliberately narrower than the
# control-plane transport's list: WITH is absent, because following a CTE to
# the statement it prefixes needs the engine's lexer, which does not exist on
# a cluster. A read that needs a CTE can be written as a subquery.
PUSHDOWN_READ_VERBS = ("SELECT", "SHOW", "DESCRIBE", "DESC", "EXPLAIN")


class SourceWriteRefused(PermissionError):
    """A statement that is not a read was handed to the pushdown transport."""


def _code_only(sql: str) -> str:
    """`sql` with string literals and comments blanked, length preserved.

    A `;` inside a literal is data, not a statement boundary, and `--` inside
    one is not a comment. Blanking rather than deleting keeps offsets, so the
    scan cannot be confused about where anything starts.

    Lexed as SNOWFLAKE lexes: a backslash escapes only inside a '...'
    string; a "..." identifier ends at the first quote that is not doubled.
    Honouring a backslash there too made `"a\\"; delete from T` one
    identifier to this scan and two statements to Snowflake -- the guard
    failed open.
    """
    out = []
    i, n = 0, len(sql)
    while i < n:
        c = sql[i]
        two = sql[i:i + 2]
        if c in ("'", '"'):
            out.append(" ")
            i += 1
            while i < n:
                if c == "'" and sql[i] == "\\" and i + 1 < n:  # escape
                    out.append("  ")
                    i += 2
                    continue
                if sql[i] == c:
                    if sql[i:i + 2] == c * 2:           # doubled = literal
                        out.append("  ")
                        i += 2
                        continue
                    out.append(" ")
                    i += 1
                    break
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            continue
        if two == "$$":
            out.append("  ")
            i += 2
            while i < n and sql[i:i + 2] != "$$":
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            out.append("  ")
            i += 2
            continue
        if two == "--":
            while i < n and sql[i] != "\n":
                out.append(" ")
                i += 1
            continue
        if two == "/*":
            depth, i = 1, i + 2
            out.append("  ")
            while i < n and depth:
                if sql[i:i + 2] == "/*":
                    depth += 1
                    out.append("  ")
                    i += 2
                elif sql[i:i + 2] == "*/":
                    depth -= 1
                    out.append("  ")
                    i += 2
                else:
                    out.append("\n" if sql[i] == "\n" else " ")
                    i += 1
            continue
        out.append(c)
        i += 1
    return "".join(out)


def assert_pushdown_read_only(sql: str) -> None:
    """Refuse anything that is not a single read. Fails closed.

    The cluster-side counterpart of the control plane's `assert_read_only`:
    the credential the notebook holds may well be able to write, and the
    only thing standing between a migration and a modified SOURCE is this
    check.
    """
    code = _code_only(sql or "")
    statements = [s for s in code.split(";") if s.strip()]
    if not statements:
        raise SourceWriteRefused(
            f"empty statement refused; this transport is read-only against "
            f"Snowflake (allowed: {', '.join(PUSHDOWN_READ_VERBS)})")
    if len(statements) > 1:
        raise SourceWriteRefused(
            f"{len(statements)} statements in one pushdown; refused. A "
            f"second statement is how a write rides along behind a read.")
    verb = statements[0].split()[0].upper() if statements[0].split() else ""
    if verb == "WITH":
        raise SourceWriteRefused(
            "a CTE is refused by this transport: deciding whether `WITH ... "
            "INSERT` is a read needs the engine's scanner, which does not "
            "run on the cluster. Write the CTE as a subquery.")
    if verb not in PUSHDOWN_READ_VERBS:
        raise SourceWriteRefused(
            f"{verb or 'unrecognised statement'} refused: this transport is "
            f"read-only against Snowflake, whatever the credential allows "
            f"(allowed: {', '.join(PUSHDOWN_READ_VERBS)}).")


class SourceConfigError(ValueError):
    """The source config is missing, unreadable or incomplete."""


def q(identifier: str) -> str:
    """Backtick-quote one Spark identifier."""
    return "`" + str(identifier).replace("`", "``") + "`"


def _sql_ident(identifier: str) -> str:
    """Double-quote one SNOWFLAKE identifier (the pushdown runs there)."""
    return '"' + str(identifier).replace('"', '""') + '"'


def _sql_literal(value: str) -> str:
    """Escape a string literal for Snowflake SQL.

    The backslash first: Snowflake reads it as an escape inside '...', so a
    table named `a\\` made `'a\\'` an unterminated literal that swallowed the
    SQL after it.
    """
    return str(value).replace("\\", "\\\\").replace("'", "''")


def load_source_config(path: str | pathlib.Path) -> dict:
    """Read the Snowflake connection config (JSON) from the workspace."""
    p = pathlib.Path(path).expanduser()
    try:
        text = p.read_text(encoding="utf-8")
    except OSError as exc:
        raise SourceConfigError(
            f"source config not readable at {p}: {exc.strerror}") from exc
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as exc:
            raise SourceConfigError(
                f"{p} is YAML but PyYAML is not on the cluster; write the "
                f"config as JSON instead") from exc
        data = yaml.safe_load(text) or {}
    else:
        data = json.loads(text) if text.strip() else {}
    if not isinstance(data, dict):
        raise SourceConfigError(f"{p}: expected a mapping at the top level")
    # THE MIGRATION CONFIG IS ONE FILE FOR BOTH ENDS: the Snowflake connection
    # nested under `snowflake:`, the AIDP coordinates under `aidp:`. That is
    # the file `provision --source-config` reads, and it uploads the
    # `snowflake:` block as JSON (`plan/<stem>.json`) -- so the shape that
    # reaches the mount is the nested one, and reading the top level for
    # `account` found nothing but the envelope key. The failure surfaced on
    # the cluster, as every required field missing at once, which reads
    # like a broken credential rather than a config one level too deep.
    nested = data.get("snowflake")
    if isinstance(nested, dict):
        return dict(nested)
    return data


class SnowflakeSource:
    """Read-only access to the Snowflake source, in either mode.

    Nothing here can write: every method issues a read, and the connector is
    read-only in AIDP 4.0 by Oracle's own statement.
    """

    def __init__(self, spark, *, mode: str = "connector",
                 config: dict | None = None,
                 external_catalog: str | None = None,
                 session_schema: str | None = None):
        if mode not in SOURCE_MODES:
            raise SourceConfigError(
                f"unknown source mode {mode!r}; expected one of "
                f"{list(SOURCE_MODES)}")
        self.spark = spark
        self.mode = mode
        self.external_catalog = external_catalog
        self._options: dict[str, str] = {}
        # A REAL schema, used only to scope a pushdown session. The connector
        # validates this option against the schemas it can see and rejects
        # INFORMATION_SCHEMA itself with DATA_ACCESS_LAYER_0031 -- but a
        # pushdown query scoped to a real schema may reference
        # INFORMATION_SCHEMA freely (both established live).
        self.session_schema = session_schema or (config or {}).get("schema")

        if mode == "external-catalog":
            if not external_catalog:
                raise SourceConfigError(
                    "external-catalog mode needs --source-catalog")
            return

        cfg = config or {}
        missing = [k for k in ("account", "warehouse", "database", "user",
                               "auth") if not cfg.get(k)]
        if missing:
            raise SourceConfigError(
                "source config is missing required field(s): "
                + ", ".join(sorted(missing)))
        account = str(cfg["account"]).strip()
        # The connector wants the account/server URL host, which is what the
        # Snowflake console calls "Account/Server URL".
        host = str(cfg.get("host")
                   or f"{account}.snowflakecomputing.com").strip()
        opts = {
            "type": "SNOWFLAKE",
            "host": host,
            "port": str(cfg.get("port") or 443),
            "database.name": str(cfg["database"]).strip(),
            "user.name": str(cfg["user"]).strip(),
            "warehouse": str(cfg["warehouse"]).strip(),
        }
        if cfg.get("role"):
            opts["role"] = str(cfg["role"]).strip()

        # A secret may be inline in the one config file, or in a file the
        # config points at. On a cluster the inline form is usually the only
        # one available, since the workspace mount carries the config but not
        # the operator's home directory.
        def secret(inline, path_field):
            if cfg.get(inline):
                return str(cfg[inline])
            if cfg.get(path_field):
                return pathlib.Path(
                    str(cfg[path_field])).expanduser().read_text(encoding="utf-8").strip()
            return None

        auth = str(cfg["auth"]).strip().lower()
        if auth == "keypair":
            key = secret("private_key", "key_path")
            if not key:
                raise SourceConfigError(
                    "auth: keypair needs `private_key` (inline) or `key_path`")
            opts["authentication.method"] = "KeyPair"
            opts["private.key.content"] = key
            passphrase = secret("key_passphrase", "key_passphrase_path")
            if passphrase:
                opts["private.key.pass.phrase"] = passphrase
        elif auth == "password":
            password = secret("password", "password_path")
            if not password:
                raise SourceConfigError(
                    "auth: password needs `password` (inline) or "
                    "`password_path`")
            opts["authentication.method"] = "Basic"
            opts["password"] = password
        else:
            raise SourceConfigError(
                f"auth {auth!r} is not supported by the AIDP Snowflake "
                f"connector; use keypair (preferred) or password")
        self._options = opts

    # -- describing the estate ------------------------------------------

    def database(self) -> str | None:
        return self._options.get("database.name")

    def pushdown(self, sql: str, *, schema: str | None = None):
        """Run `sql` IN SNOWFLAKE and return a DataFrame.

        Refuses anything that is not a single read statement, whatever the
        credential allows -- see `assert_pushdown_read_only`.

        Connector mode only. The `schema` option must name a REAL schema:
        the connector rejects INFORMATION_SCHEMA there with
        DATA_ACCESS_LAYER_0031, while a query scoped to a real schema may
        reference INFORMATION_SCHEMA freely. Both established live.
        """
        # Before the mode check and before anything reaches Spark: the
        # refusal must not depend on configuration being right.
        assert_pushdown_read_only(sql)
        if self.mode != "connector":
            raise SourceConfigError(
                "pushdown is connector-mode only; external-catalog mode has "
                "no Snowflake session to push into")
        scope = schema or self.session_schema
        if not scope:
            raise SourceConfigError(
                "connector pushdown needs a REAL schema to scope the "
                "session: add `schema:` to the source config or pass "
                "--session-schema. INFORMATION_SCHEMA is not accepted there.")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", scope)
                .option("pushdown.sql", sql)
                .load())

    def source_counts(self, schema: str, tables: list[str], *,
                      chunk: int = 50) -> dict[str, int]:
        """`{table: COUNT(*)}` for many tables in ONE round trip per chunk.

        Connector mode opens a Snowflake session per read, and the copy needs
        a source count twice per table (before, and again after, to catch a
        source that moved during the copy). Per-table counting therefore cost
        more session setup than the copy itself on a live run. A single
        UNION ALL answers a whole schema instead; it is chunked because a
        statement with thousands of branches is its own problem.

        External-catalog mode has no session to amortise, so it falls back to
        a count per table -- correct either way, and the caller does not care.
        """
        out: dict[str, int] = {}
        if self.mode != "connector":
            for table in tables:
                out[table] = self.read_table(schema, table).count()
            return out
        for start in range(0, len(tables), chunk):
            batch = tables[start:start + chunk]
            sql = " union all ".join(
                # The literal is the table's own name, so one query can carry
                # many counts and still say which is which. Both the literal
                # and the identifier are escaped: one apostrophe in a table
                # name would otherwise break the whole chunk.
                f"select '{_sql_literal(t)}' as SNOWMIG_TABLE, "
                f"count(*) as SNOWMIG_N from {_sql_ident(t)}"
                for t in batch)
            for row in self.pushdown(sql, schema=schema).collect():
                data = row.asDict()
                out[str(data["SNOWMIG_TABLE"])] = int(data["SNOWMIG_N"])
        return out

    def read_table(self, schema: str, table: str):
        """A DataFrame over one source table."""
        if self.mode == "external-catalog":
            return self.spark.table(
                f"{q(self.external_catalog)}.{q(schema)}.{q(table)}")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", schema)
                .option("table", table)
                .load())

    def register_temp_view(self, schema: str, table: str, view: str) -> str:
        """Expose a source table to SQL as a temp view, and return its name.

        INSERT ... SELECT needs the source addressable in SQL. In
        external-catalog mode the three-part name already is; in connector
        mode the DataFrame is registered as a session-local temp view, which
        is dropped by the caller.
        """
        if self.mode == "external-catalog":
            return f"{q(self.external_catalog)}.{q(schema)}.{q(table)}"
        self.read_table(schema, table).createOrReplaceTempView(view)
        return q(view)

    def drop_temp_view(self, view: str) -> None:
        if self.mode == "connector":
            self.spark.catalog.dropTempView(view)

    def describe(self) -> dict:
        """What this source is, for the report header. Never the credential."""
        out = {"mode": self.mode}
        if self.mode == "external-catalog":
            out["external_catalog"] = self.external_catalog
        else:
            out.update(session_schema=self.session_schema,
                       host=self._options.get("host"),
                       database=self._options.get("database.name"),
                       user=self._options.get("user.name"),
                       warehouse=self._options.get("warehouse"),
                       role=self._options.get("role"),
                       auth=self._options.get("authentication.method"))
        return out


## Stage logic

In [ ]:
#!/usr/bin/env python3
"""Create target schemas and EMPTY Delta tables for one schema (or all).

Runs on AIDP compute. Three structure sources, chosen with --mode:

  ddl-plan (default)  types come from `plan/ddl_plan.json`, which the
                      migrator's own type mapper produced -- it refuses what
                      it cannot map exactly instead of guessing, and the plan
                      was reviewed and signed off before this ran. Costs NO
                      source read per table, which is what makes a large
                      estate feasible: creating 100 tables by CTAS took
                      minutes on a live run, because each CTAS is its own
                      Snowflake round trip.
  ctas                CREATE TABLE ... USING DELTA AS SELECT * ... WHERE 1=0.
                      Spark derives the types through the connector, so the
                      copy cannot hit a type the table cannot hold -- but the
                      mapping is the connector's, not an audited one, and it
                      pays a source read per table.
  manifest            types come from discovery_manifest.json verbatim. Only
                      valid when the manifest carries SPARK types, i.e. it
                      was built in external-catalog mode (DESCRIBE); a
                      connector-mode manifest carries SNOWFLAKE types and the
                      whole run is refused before anything is created. Some
                      of those types Delta accepts verbatim with another
                      meaning -- FLOAT is 64-bit in Snowflake and 32-bit in
                      Spark -- so no per-type check can tell them apart.

Safety: CREATE TABLE IF NOT EXISTS everywhere; nothing is ever dropped or
replaced here. The target catalog must be INTERNAL — this script REFUSES to
address the source catalog as its target, and AIDP refuses DDL on external
catalogs anyway (documented), so the failure would be loud, not silent.

The CREATE returning is not the claim: IF NOT EXISTS is a silent no-op on a
table that is already there, so every table is DESCRIBEd afterwards and
compared with the plan, column by column and in order.

In --mode ddl-plan the plan's `NOT NULL`, column COMMENTs and table COMMENT
are applied, not just its names and types: they are in the CREATE TABLE the
reviewer approved, and this stage used to render `name type` and then compare
against the same reduced shape, so a table that differed from the approved
SQL still read back as matching. Nullability is read from the table's schema
(DESCRIBE does not report it); when that read fails the table's record says
the property is UNCHECKED rather than counting it as applied.

Writes `structure_report_<schema>.json` per schema, one status per table:
  created          it was not there before, and it reads back as planned
  already_existed  it was there before, and it matches the plan (in
                   --mode ctas there is no plan: the layout is NOT
                   compared, and the record's reason says so)
  type_drift       it was there with a layout the plan did not produce; it
                   is left as found, listed with the differing columns, and
                   counted as a problem (exit 1) -- the copy is a positional
                   INSERT, so a mismatched layout would land rows in the
                   wrong columns with matching counts
  not_in_plan      the approved plan carries no columns for it; NOT created
  failed           the CREATE raised; the error is the reason
Resumable: `created` and `already_existed` are skipped on a re-run with the
same --mode (--force re-checks them); one recorded under another --mode is
re-checked, because that mode's layout is not this one's -- a table --mode
manifest created is not thereby what the ddl plan approved. `type_drift`,
`failed` and `not_in_plan` are looked at again every run, so fixing the
table or the plan is enough.
"""
from __future__ import annotations

import argparse
import datetime
import json
import pathlib
import sys


# /Workspace is the live-verified mount of the workspace tree on cluster
# filesystems (probed 2026-09-16 on a real cluster).
DEFAULT_REPORTS_DIR = "/Workspace/backup-snowflake-migration/reports"
MANIFEST_NAME = "discovery_manifest.json"


def three(*parts: str) -> str:
    return ".".join(q(p) for p in parts)


def log(msg: str) -> None:
    print(f"[structure] {msg}", flush=True)


def fail(msg: str) -> int:
    """Report a refusal on BOTH streams and return 1.

    A notebook task captures stdout only: live, a script that exited 1 via a
    stderr-only message produced a job failure with NO explanation anywhere.
    """
    print(f"ERROR: {msg}", flush=True)
    print(f"error: {msg}", file=sys.stderr)
    return 1


def _report_path(reports: pathlib.Path, schema: str) -> pathlib.Path:
    return reports / f"structure_report_{schema.lower()}.json"


def _load_report(path: pathlib.Path, schema: str, target: str) -> dict:
    """The prior report for this schema, but ONLY if it targeted the same place.

    Resumability is keyed by SOURCE schema, so a report written while
    targeting one destination would otherwise let a run against a DIFFERENT
    destination skip every create as "already created" -- observed live, with
    an empty target schema reported as done. A changed target starts a fresh
    record and says so.
    """
    if not path.exists():
        return {"schema": schema, "objects": {}, "target": target}
    prior = json.loads(path.read_text(encoding="utf-8"))
    if prior.get("target") and prior["target"] != target:
        log(f"{schema}: the previous report targeted {prior['target']}, not "
            f"{target} — starting a fresh record for this target (the old "
            f"one is kept at {path.name}.{prior['target'].replace('.', '_')})")
        path.with_suffix(
            f".{prior['target'].replace('.', '_')}.json").write_text(
                json.dumps(prior, indent=2), encoding="utf-8")
        return {"schema": schema, "objects": {}, "target": target}
    return prior


class TypeDrift(Exception):
    """The table was already there with a layout the plan did not produce."""


def _describe_columns(spark, fqn: str) -> list[tuple[str, str]] | None:
    """(name, type) pairs from DESCRIBE, or None when the table is not there.

    Columns end at the first blank or `#` row (Delta's metadata section).
    """
    try:
        rows = spark.sql(f"DESCRIBE {fqn}").collect()
    except Exception:
        return None
    out: list[tuple[str, str]] = []
    for row in rows:
        name = str(row["col_name"] or "").strip()
        if not name or name.startswith("#"):
            break
        out.append((name, str(row["data_type"] or "")))
    return out


def _describe_comments(spark, fqn: str) -> dict[str, str] | None:
    """{column: comment} from DESCRIBE's third column, upper-cased keys.

    None when the read-back carries no comment column at all -- a comment
    that was NOT LOOKED AT must not read as a comment that is missing.
    DESCRIBE reports comments; it does NOT report nullability, which is why
    that is read from the table's schema instead.
    """
    try:
        rows = spark.sql(f"DESCRIBE {fqn}").collect()
    except Exception:
        return None
    out: dict[str, str] = {}
    carried = False
    for row in rows:
        name = str(row["col_name"] or "").strip()
        if not name or name.startswith("#"):
            break
        try:
            value = row["comment"]
            carried = True
        except Exception:
            value = None
        out[name.upper()] = str(value or "")
    return out if carried else None


def _nullability(spark, fqn: str) -> dict[str, bool] | None:
    """{column: nullable} from the table's schema, or None when unreadable.

    DESCRIBE has no nullability column, so the read-back for `NOT NULL` is
    the StructType. None means NOT CHECKED -- reported as such rather than
    passed off as a match, because "we did not look" and "it is right" are
    the two answers this whole stage exists to keep apart.
    """
    try:
        fields = spark.table(fqn).schema.fields
        return {str(f.name).upper(): bool(f.nullable) for f in fields}
    except Exception:
        return None


def _norm_type(value: str) -> str:
    """Compare types ignoring case and internal spacing only."""
    return "".join(str(value).split()).upper()


def _compare_columns(expected: list[dict],
                     actual: list[tuple[str, str]],
                     nullable: dict[str, bool] | None = None,
                     comments: dict[str, str] | None = None) -> str | None:
    """None if the structures match, else a one-line description of the diff.

    Same rule as the control-plane deploy: names and types, in order. A
    same-count layout in another order is a diff -- the copy is a positional
    INSERT, so that is the case that lands rows in the wrong columns with
    matching counts.

    `NOT NULL` and column COMMENTs are part of the approved DDL, so they are
    compared too when the read-back supplies them: a table created with the
    reviewed SQL and reported "verified" against a name-and-type-only
    comparison is how the reviewed artifact and the applied one came apart in
    the first place. `nullable=None` means the schema could not be read; the
    caller says so rather than counting it as a match.
    """
    want = [(str(c.get("name", "")).upper(), _norm_type(c.get("type", "")))
            for c in expected]
    got = [(n.upper(), _norm_type(ty)) for n, ty in actual]
    if want != got:
        if len(want) != len(got):
            return (f"column count differs: planned {len(want)}, found {len(got)} "
                    f"(planned {[n for n, _ in want]}, found {[n for n, _ in got]})")
        diffs = [f"position {i + 1}: planned {w[0]} {w[1]}, found {g[0]} {g[1]}"
                 for i, (w, g) in enumerate(zip(want, got)) if w != g]
        return "; ".join(diffs)

    property_diffs: list[str] = []
    for col in expected:
        name = str(col.get("name", "")).upper()
        if nullable is not None and col.get("nullable") is False \
                and nullable.get(name, True):
            property_diffs.append(
                f"{name}: planned NOT NULL, found nullable")
        if comments is not None:
            planned = str(col.get("description") or "")
            found = str(comments.get(name) or "")
            if planned and planned != found:
                property_diffs.append(
                    f"{name}: planned comment {planned!r}, found {found!r}")
    return "; ".join(property_diffs) or None


def create_table_ctas(source: SnowflakeSource, schema: str, name: str,
                      target_catalog: str, target_schema: str) -> str:
    """Empty table whose columns Spark derives from the SOURCE read.

    The source is addressed through `SnowflakeSource`, so this works in
    connector mode (a temp view over the connector read) as well as against
    an external catalog's three-part name. Returns `created`, or
    `already_existed` when the table was there before this run -- CTAS has
    no plan to compare that layout with, so it is reported, not checked.
    """
    fqn = three(target_catalog, target_schema, name)
    if _describe_columns(source.spark, fqn) is not None:
        return "already_existed"
    view = f"snowmig_src_{schema}_{name}".lower()[:120]
    ref = source.register_temp_view(schema, name, view)
    try:
        source.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {fqn} "
            f"USING DELTA AS SELECT * FROM {ref} WHERE 1=0")
    finally:
        source.drop_temp_view(view)
    return "created"


def lit(value: str) -> str:
    """A single-quoted Spark string literal, escaped the way Spark expects.

    Backslash, not doubling: Spark reads `'it\\'\\'s'` as two adjacent
    literals and concatenates them, so a doubled quote silently eats the
    apostrophe. Identical to `target.ddl.quote_spark_string`, which this
    stage cannot import (it is uploaded as a single standalone file).
    """
    escaped = str(value).replace("\\", "\\\\").replace("'", "\\'")
    return "'" + escaped + "'"


def _column_sql(col: dict) -> str:
    """One column of the CREATE TABLE, from one `expected_columns` entry.

    The SAME rules as `target.ddl.render_column_sql`, which wrote the SQL the
    operator approved in DDL_PLAN.md -- a parity test in the engine's suite
    holds the two together. This stage used to render `name type` only, so
    the `NOT NULL` and the COMMENT in the approved SQL were dropped here and
    the comparison below then agreed with itself.
    """
    piece = f'{q(col["name"])} {col["type"]}'
    if col.get("nullable") is False:
        piece += " NOT NULL"
    if col.get("description"):
        piece += " COMMENT " + lit(col["description"])
    return piece


def create_table_from_columns(spark, columns: list[dict],
                              target_catalog: str, target_schema: str,
                              name: str, description: str = "",
                              notes: list | None = None) -> str:
    """CREATE TABLE from an explicit column list, then READ IT BACK.

    Types are used verbatim, and so are the plan's `nullable` and
    `description` -- the properties the approved SQL shows. `CREATE TABLE IF
    NOT EXISTS` is a silent no-op on a table that is already there, so the
    CREATE returning is not the claim: the table is DESCRIBEd afterwards and
    compared with the plan, nullability included.
    Returns `created` (it was not there before and now matches),
    `already_existed` (it was there and matches), or raises TypeDrift when
    what is there differs from the plan -- the table is left as found.
    `notes` collects what could NOT be checked, so an unverified property is
    never reported as a verified one.
    """
    if not columns:
        raise ValueError("no column list for this table; rediscover it or "
                         "use --mode ctas")
    fqn = three(target_catalog, target_schema, name)
    before = _describe_columns(spark, fqn)
    if before is None:
        cols = ", ".join(_column_sql(c) for c in columns)
        spark.sql(f"CREATE TABLE IF NOT EXISTS {fqn} ({cols}) USING DELTA"
                  + (f" COMMENT {lit(description)}" if description else ""))
        after = _describe_columns(spark, fqn)
        if after is None:
            raise RuntimeError("CREATE TABLE returned but the table does not "
                               "DESCRIBE afterwards; NOT created")
    else:
        after = before

    # Both extra read-backs are skipped when the plan asks for nothing they
    # would check: a table with no NOT NULL and no comments costs exactly
    # what it cost before.
    wants_not_null = any(c.get("nullable") is False for c in columns)
    wants_comments = any(c.get("description") for c in columns)
    nullable = _nullability(spark, fqn) if wants_not_null else None
    comments = _describe_comments(spark, fqn) if wants_comments else None
    # Not a match and not a failure: it was not looked at. Said out loud,
    # because a property reported as applied when nobody checked is the
    # defect this stage is guarding against.
    if notes is not None:
        if wants_not_null and nullable is None:
            notes.append(
                "NOT NULL was requested but could not be verified: the "
                "table's schema could not be read back, so nullability is "
                "UNCHECKED on this table")
        if wants_comments and comments is None:
            notes.append(
                "column COMMENTs were requested but DESCRIBE carried no "
                "comment column, so they are UNCHECKED on this table")
    diff = _compare_columns(columns, after, nullable, comments)
    if diff is None:
        return "created" if before is None else "already_existed"
    if before is None:
        raise TypeDrift(f"created by this run, but it reads back differently "
                        f"from the plan -- {diff}")
    raise TypeDrift(f"already there with a layout the plan did not produce -- "
                    f"{diff}. CREATE TABLE IF NOT EXISTS left it as found; "
                    f"NOT created from the plan")


# Types Snowflake reports but Spark/Delta does not accept verbatim. Their
# presence in a manifest means it was built in CONNECTOR mode, where the
# manifest records SOURCE types on purpose -- translating them here would
# duplicate (and inevitably diverge from) the migrator's own type mapper,
# which refuses ambiguous cases rather than guessing.
_SNOWFLAKE_ONLY_TYPES = ("NUMBER", "TEXT", "VARIANT", "OBJECT", "GEOGRAPHY",
                         "GEOMETRY", "TIMESTAMP_LTZ", "TIMESTAMP_TZ")


def _looks_like_snowflake_types(columns: list[dict]) -> bool:
    return any(str(c.get("type", "")).upper().startswith(t)
               for c in columns for t in _SNOWFLAKE_ONLY_TYPES)


def _snowflake_typed_manifest(manifest: dict, schemas: list[str]) -> str | None:
    """Why this manifest's types are SNOWFLAKE types, or None.

    Decided for the manifest, not per table. The prefix list above only
    catches types Delta rejects; FLOAT, DATE and BOOLEAN pass it, and FLOAT
    is then created 32-bit where Snowflake's is a double: READINGS(READING
    FLOAT) was recorded `created` and the copy narrowed every value to ~7
    digits. Discovery records which mode wrote the manifest, and the
    connector's raw `data_type` field is never written by DESCRIBE.
    """
    mode = (manifest.get("source") or {}).get("mode")
    if mode and mode != "external-catalog":
        return f"it was built in {mode} mode"
    by_name = {s.get("name"): s for s in manifest.get("schemas") or []}
    for schema in schemas:
        for table in (by_name.get(schema) or {}).get("tables") or []:
            if any("data_type" in c for c in table.get("columns") or []):
                return (f"{schema}.{table['name']} carries the connector's "
                        f"raw `data_type` field")
    return None


def columns_from_ddl_plan(ddl_plan: dict) -> dict[tuple[str, str], list[dict]]:
    """{(source_schema, table): [{name, type}]} from the engine's ddl_plan.

    The engine translated these types with its full discipline (it blocks a
    table it cannot map exactly), and the plan was the artifact the user
    signed off, so applying it needs no source read and no cluster-side
    translation.
    """
    out: dict[tuple[str, str], list[dict]] = {}
    for stmt in ddl_plan.get("statements") or []:
        ident = str(stmt.get("source_identifier") or "")
        parts = ident.split(".")
        if len(parts) != 3 or not stmt.get("expected_columns"):
            continue
        if str(stmt.get("object_type") or "TABLE").upper() == "VIEW":
            continue                    # views are not created by this path
        out[(parts[1], parts[2])] = stmt["expected_columns"]
    return out


def targets_from_ddl_plan(ddl_plan: dict
                          ) -> dict[tuple[str, str], tuple[str, str]]:
    """{(source_schema, table): (target_schema, target_name)} from the plan.

    The plan's `target_fqn` IS the approved name. Deriving it again from the
    source schema silently discards `--bronze-catalog-prefix` and
    `--bronze-schema-style`, and creates an object the reviewer never saw.

    A statement without a three-part `target_fqn` contributes nothing: this
    map is only ever used to place an object the plan actually named.
    """
    out: dict[tuple[str, str], tuple[str, str]] = {}
    for stmt in ddl_plan.get("statements") or []:
        source = str(stmt.get("source_identifier") or "").split(".")
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(source) != 3 or len(target) != 3:
            continue
        if str(stmt.get("object_type") or "TABLE").upper() == "VIEW":
            continue
        out[(source[1], source[2])] = (target[1], target[2])
    return out


def catalogs_from_ddl_plan(ddl_plan: dict) -> set:
    """Every catalog the plan targets. More than one, or one that is not
    the catalog this run was given, is the operator's to see."""
    out = set()
    for stmt in ddl_plan.get("statements") or []:
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(target) == 3:
            out.add(target[0])
    return out


def descriptions_from_ddl_plan(ddl_plan: dict) -> dict[tuple[str, str], str]:
    """{(source_schema, table): table COMMENT} from the engine's ddl_plan.

    The source table's COMMENT is in the approved CREATE TABLE, so it is
    applied here too rather than being the one property the reviewer sees
    and the target never gets.
    """
    out: dict[tuple[str, str], str] = {}
    for stmt in ddl_plan.get("statements") or []:
        parts = str(stmt.get("source_identifier") or "").split(".")
        if len(parts) != 3 or not stmt.get("description"):
            continue
        if str(stmt.get("object_type") or "TABLE").upper() == "VIEW":
            continue
        out[(parts[1], parts[2])] = str(stmt["description"])
    return out


def views_from_ddl_plan(ddl_plan: dict) -> set[tuple[str, str]]:
    """{(source_schema, view)} for every VIEW statement in the plan."""
    out: set[tuple[str, str]] = set()
    for stmt in ddl_plan.get("statements") or []:
        parts = str(stmt.get("source_identifier") or "").split(".")
        if len(parts) == 3 and str(stmt.get("object_type") or "").upper() == "VIEW":
            out.add((parts[1], parts[2]))
    return out


# This stage creates TABLES. The plan's CREATE VIEW statements are qualified
# with the plan-time target and their bodies would need re-qualifying for a
# run-time --target-catalog, so views are the catalog path's job (`snowmig
# deploy --execute`). They are still LISTED here, so a view the plan promised
# can never be absent from every report with exit 0.
VIEW_NOT_CREATED = ("views are not created by this stage (tables only); "
                    "create them with `snowmig deploy --execute` and verify "
                    "them against the source")


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--source-mode", choices=list(SOURCE_MODES),
                    default="connector",
                    help="how to READ the source when --mode ctas derives "
                         "types from it (see snowmig_source.py)")
    ap.add_argument("--source-config",
                    help="JSON/YAML connection config (connector mode)")
    ap.add_argument("--source-catalog",
                    help="the registered EXTERNAL catalog "
                         "(external-catalog mode)")
    ap.add_argument("--target-catalog", required=True)
    ap.add_argument("--schema", action="append", default=None,
                    help="repeatable; default: every schema in the manifest")
    ap.add_argument("--target-schema", default=None,
                    help="override the target schema name (single --schema "
                         "runs only); default mirrors the source")
    ap.add_argument("--mode", choices=("ddl-plan", "ctas", "manifest"),
                    default="ddl-plan")
    ap.add_argument("--ddl-plan",
                    help="path to ddl_plan.json (default: ../plan/"
                         "ddl_plan.json next to --reports-dir)")
    ap.add_argument("--reports-dir", default=DEFAULT_REPORTS_DIR)
    ap.add_argument("--dry-run", action="store_true",
                    help="print every statement; execute nothing")
    ap.add_argument("--force", action="store_true",
                    help="re-check tables the report already records as "
                         "created or already_existed")
    args = ap.parse_args(argv)

    if args.source_catalog and \
            args.source_catalog.lower() == args.target_catalog.lower():
        return fail("error: source and target catalog are the same. The source is "
              "the read-only EXTERNAL catalog; the target must be an INTERNAL "
              "one.")
    if args.target_schema and len(args.schema or []) != 1:
        return fail("error: --target-schema needs exactly one --schema")

    reports = pathlib.Path(args.reports_dir)
    manifest = json.loads((reports / MANIFEST_NAME).read_text(encoding="utf-8"))
    by_name = {s["name"]: s for s in manifest["schemas"]}
    schemas = args.schema or sorted(by_name)

    if args.mode == "manifest":
        why = _snowflake_typed_manifest(manifest, schemas)
        if why:
            return fail(
                f"error: --mode manifest needs a manifest of SPARK types, and "
                f"this one records SNOWFLAKE types ({why}; 00_discover's "
                f"default connector mode writes them on purpose). Delta "
                f"rejects some of them and accepts others with a different "
                f"meaning -- FLOAT would be created 32-bit -- so nothing was "
                f"created. Use --mode ddl-plan (engine-translated types) or "
                f"--mode ctas.")

    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    planned_columns: dict = {}
    planned_descriptions: dict = {}
    planned_views: set | None = None
    planned_targets: dict = {}
    if args.mode == "ddl-plan":
        ddl_path = (pathlib.Path(args.ddl_plan) if args.ddl_plan
                    else reports.parent / "plan" / "ddl_plan.json")
        if not ddl_path.is_file():
            return fail(f"--mode ddl-plan needs ddl_plan.json; {ddl_path} is "
                        f"not there. Run the migrator's `ddl` stage and let "
                        f"`provision` upload it, or pass --ddl-plan")
        ddl_plan = json.loads(ddl_path.read_text(encoding="utf-8"))
        planned_columns = columns_from_ddl_plan(ddl_plan)
        planned_descriptions = descriptions_from_ddl_plan(ddl_plan)
        planned_views = views_from_ddl_plan(ddl_plan)
        planned_targets = targets_from_ddl_plan(ddl_plan)
        # A plan for another catalog is a different migration. Creating its
        # tables here under this run's catalog would be the same silent
        # substitution this map exists to stop.
        plan_catalogs = catalogs_from_ddl_plan(ddl_plan)
        stray = {c for c in plan_catalogs
                 if c.lower() != str(args.target_catalog).lower()}
        if stray:
            return fail(
                f"error: the approved plan targets catalog(s) "
                f"{', '.join(sorted(stray))}, and this run was given "
                f"--target-catalog {args.target_catalog}. Creating the "
                f"plan's tables somewhere it does not name would be exactly "
                f"the substitution the plan exists to prevent. Re-run `plan "
                f"--bronze-catalog-prefix {args.target_catalog}` and `ddl` "
                f"so the plan names this catalog, or point this run at the "
                f"one the plan names.")
        log(f"ddl plan: {len(planned_columns)} table(s) with engine-"
            f"translated types, from {ddl_path}"
            + (f"; {len(planned_views)} view(s) it carries are NOT created "
               f"by this stage" if planned_views else ""))

    source = None
    if args.mode == "ctas":
        try:
            config = (load_source_config(args.source_config)
                      if args.source_config else None)
            source = SnowflakeSource(spark, mode=args.source_mode,
                                     config=config,
                                     external_catalog=args.source_catalog)
        except SourceConfigError as exc:
            return fail(f"error: {exc}")

    failures = 0
    # Run-wide, not per schema: a canary plan scoped to one schema
    # legitimately leaves every other schema all-`not_in_plan`.
    created_total = 0
    not_in_plan_total = 0
    for schema in schemas:
        record = by_name.get(schema)
        if record is None:
            return fail(f"error: schema {schema!r} is not in the manifest; run "
                  f"00_discover first")
        # The approved plan decides the target namespace. Only where the
        # plan is silent (ctas/manifest mode, or a table it does not carry)
        # does the source schema name stand in.
        planned_for_schema = {
            tgt_schema for (src_schema, _t), (tgt_schema, _n)
            in planned_targets.items() if src_schema == schema}
        if args.target_schema:
            target_schema = args.target_schema
            if planned_for_schema and target_schema not in planned_for_schema:
                return fail(
                    f"error: --target-schema {target_schema!r} contradicts "
                    f"the approved plan, which puts {schema} in "
                    f"{', '.join(sorted(planned_for_schema))}. The plan is "
                    f"the reviewed artifact; change it, or drop the flag.")
        elif len(planned_for_schema) == 1:
            target_schema = next(iter(planned_for_schema))
        elif len(planned_for_schema) > 1:
            return fail(
                f"error: the approved plan puts source schema {schema} in "
                f"more than one target schema "
                f"({', '.join(sorted(planned_for_schema))}); this stage "
                f"creates one schema per run. Pass --target-schema to say "
                f"which.")
        else:
            target_schema = schema
        path = _report_path(reports, schema)
        target = f"{args.target_catalog}.{target_schema}"
        report = _load_report(path, schema, target)
        # Which mode wrote the statuses loaded above, for records that
        # predate the per-object `mode`.
        prior_mode = report.get("mode")
        report["target"] = target
        report["mode"] = args.mode

        schema_sql = (f"CREATE SCHEMA IF NOT EXISTS "
                      f"{q(args.target_catalog)}.{q(target_schema)}")
        if args.dry_run:
            log(f"DRY RUN: {schema_sql}")
        else:
            spark.sql(schema_sql)

        for table in record["tables"]:
            name = table["name"]
            notes: list[str] = []
            prior = report["objects"].get(name, {})
            done = prior.get("status") in ("created", "already_existed")
            # Every report this stage writes names its mode; one that does
            # not was not written by it, and is taken as this run's.
            recorded_by = prior.get("mode", prior_mode) or args.mode
            if done and not args.force and recorded_by == args.mode:
                log(f"skip {schema}.{name}: already {prior['status']}")
                created_total += 1
                continue
            if done and not args.force:
                # Another mode's `created` checked that mode's layout, not
                # this one's: a FLOAT table --mode manifest made was skipped
                # here as done, with the plan saying DOUBLE.
                log(f"re-check {schema}.{name}: recorded {prior['status']} "
                    f"by --mode {recorded_by}, not {args.mode}")
            try:
                if args.dry_run:
                    log(f"DRY RUN: would create "
                        f"{args.target_catalog}.{target_schema}.{name} "
                        f"({args.mode})")
                    status = "dry_run"
                elif args.mode == "ctas":
                    status = create_table_ctas(source, schema, name,
                                               args.target_catalog,
                                               target_schema)
                elif args.mode == "ddl-plan":
                    columns = planned_columns.get((schema, name))
                    if not columns:
                        report["objects"][name] = {
                            "status": "not_in_plan",
                            "reason": "the approved ddl_plan carries no "
                                      "columns for this table -- the engine "
                                      "either blocked it or it was outside "
                                      "the plan's scope. NOT created."}
                        path.write_text(json.dumps(report, indent=2), encoding="utf-8")
                        log(f"{schema}.{name}: not in the approved plan")
                        not_in_plan_total += 1
                        continue
                    status = create_table_from_columns(
                        spark, columns, args.target_catalog, target_schema,
                        name,
                        description=planned_descriptions.get((schema, name),
                                                             ""),
                        notes=notes)
                else:
                    columns = table.get("columns") or []
                    if _looks_like_snowflake_types(columns):
                        raise ValueError(
                            "this manifest carries SNOWFLAKE types (it was "
                            "built in connector mode), which Delta will not "
                            "accept verbatim. Use --mode ddl-plan (engine-"
                            "translated types) or --mode ctas.")
                    status = create_table_from_columns(
                        spark, columns, args.target_catalog, target_schema,
                        name, notes=notes)
                report["objects"][name] = {"status": status,
                                           "mode": args.mode}
                if notes:
                    # Properties that could NOT be read back. Recorded next
                    # to the status so "created" never implies "and every
                    # property was checked".
                    report["objects"][name]["unverified_properties"] = notes
                if args.mode == "ctas" and status == "already_existed":
                    # CTAS has no plan to compare the layout with: the
                    # table was there before this run and nobody has
                    # checked it. Said here, so the copy scope and the
                    # reconcile report carry it rather than a bare pass.
                    report["objects"][name]["reason"] = (
                        "there before this run; --mode ctas has no plan "
                        "to compare its layout with, so the layout was "
                        "NOT compared")
                log(f"{schema}.{name}: {status}")
                if status in ("created", "already_existed"):
                    created_total += 1
            except TypeDrift as exc:
                # A problem state, not a failure of THIS run: the table is
                # there, it is not what the plan says, and a positional copy
                # into it would land rows in the wrong columns with matching
                # counts. Re-checked on every run until it matches.
                failures += 1
                report["objects"][name] = {"status": "type_drift",
                                           "reason": str(exc)[:400]}
                log(f"{schema}.{name}: TYPE DRIFT — {str(exc)[:200]}")
            except Exception as exc:
                failures += 1
                report["objects"][name] = {"status": "failed",
                                           "reason": str(exc)[:400]}
                log(f"{schema}.{name}: FAILED — {str(exc)[:200]}")
            report["updated_at"] = datetime.datetime.now(
                datetime.timezone.utc).isoformat()
            path.write_text(json.dumps(report, indent=2), encoding="utf-8")

        # Views: listed, never created here. Kept apart from `objects` so
        # the table tally, the resume logic and the copy scope stay table-only.
        views = [v["name"] for v in record.get("views") or []]
        if views:
            report["views"] = {}
            for view in views:
                entry = {"status": "not_created_by_this_path",
                         "reason": VIEW_NOT_CREATED}
                if planned_views is not None:
                    entry["in_plan"] = (schema, view) in planned_views
                report["views"][view] = entry
            path.write_text(json.dumps(report, indent=2), encoding="utf-8")
            log(f"{schema}: {len(views)} view(s) in the manifest are NOT "
                f"created by this stage ({', '.join(views[:5])}"
                f"{', ...' if len(views) > 5 else ''}); use `snowmig deploy "
                f"--execute` for views")

        counts = {}
        for obj in report["objects"].values():
            counts[obj["status"]] = counts.get(obj["status"], 0) + 1
        log(f"{schema}: {counts} -> {path}")

    log(f"run: created or already there {created_total}, not in plan "
        f"{not_in_plan_total}, failed or drifted {failures}")
    if failures:
        return 1
    if args.mode == "ddl-plan" and not args.dry_run and not created_total \
            and not_in_plan_total:
        # Every per-table record above is right; the RUN still did nothing.
        # Exit 0 here gave three SUCCESS jobs (structure, copy, reconcile)
        # for a plan that never overlapped the requested schema.
        return fail(f"error: created 0 table(s); {not_in_plan_total} were not "
                    f"in the approved plan ({ddl_path}). The plan and the "
                    f"requested schema(s) do not overlap -- is this the "
                    f"ddl_plan.json for THIS estate and wave? Nothing was "
                    f"created, so 02_copy_schema has nothing to copy.")
    return 0



In [ ]:
# ── RUN ────────────────────────────────────────────────────────────
# main() RETURNS an exit code; it is not allowed to raise SystemExit here.
# A notebook cell that raises SystemExit is reported as a FAILED task even
# when the work succeeded, so the code is inspected and only a real failure
# is re-raised -- which keeps a genuinely failed stage failing.
code = main(ARGV)
print('exit code:', code, flush=True)
if code:
    raise RuntimeError(f'stage exited {code}')
